# Exercise 7 · Run the tests on every pull request

Codex writes a workflow that runs your tests on GitHub for every pull request.

**Goal:** A workflow runs pytest on each pull request and each push to `main`.

**Why it matters:** GitHub runs your tests on a clean machine, which lacks what your laptop has.

**Before you start:** You did Tuesday's exercises: commit, push, pull request, merge. You installed Codex and signed in.

**What the check measures:** `checks.yml` has the lines of Step 5. Your newest pull request passed `checks`, and you merged it.

**Done when:** the last cell prints `Done:`.

## The situation

`cases.py` holds `filter_cases`, which picks the cases with one status. `tests/test_cases.py` has two tests for it. The files hold made-up data.

GitHub Actions runs a workflow: a file in `.github/workflows/`. Each job in it runs "in a fresh instance of a runner image" ([Workflow syntax](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax)): a new machine, without what your laptop supplied. A pull request shows each job as a check, named after the job.

## Step 1 · Write the practice files

Run the cell. It writes five files into `practice/` and never overwrites one. `requirements.txt` names pytest for the clean machine.

Expect five lines that start with `Wrote practice/` or `Kept`.

In [ ]:
FILES = {
    "cases.py": """\"\"\"Pick the cases that have one status.\"\"\"


def filter_cases(cases: list[dict], status: str) -> list[dict]:
    \"\"\"Return the cases that have one status, in their order.

    Args:
        cases: The cases. Each one is a dict with a "status" key.
        status: The status to keep.

    Returns:
        The cases whose status is status, in their order.
    \"\"\"
    return [case for case in cases if case["status"] == status]
""",
    "tests/test_cases.py": """from cases import filter_cases

CASES = [
    {"id": "C-1", "status": "open"},
    {"id": "C-2", "status": "open"},
    {"id": "C-3", "status": "closed"},
]


def test_open_returns_only_open_cases() -> None:
    assert [case["id"] for case in filter_cases(CASES, "open")] == ["C-1", "C-2"]


def test_unknown_status_returns_no_cases() -> None:
    assert filter_cases(CASES, "pending") == []
""",
    "requirements.txt": """pytest==9.1.0
""",
    "pytest.ini": """[pytest]
pythonpath = .
""",
    ".gitignore": """__pycache__/
""",
}

from pathlib import Path

PRACTICE = Path("practice")
for name, text in FILES.items():
    path = PRACTICE / name
    if path.exists():
        print(f"Kept practice/{name}: it is already there.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(text, encoding="utf-8")
        print(f"Wrote practice/{name}")

## Step 2 · Run the tests and commit the start

Choose **Terminal > New Terminal**. Type each line, then press Enter:

```sh
source .venv/bin/activate
cd week-02/day-03/exercise-07-tests-on-every-pull-request/practice
python -m pytest -q
git init -q -b main && git add -A && git commit -qm "Start"
```

Expect `2 passed`, then no output from the last line.

## Step 3 · Make an empty public repository and push main

1. On github.com, click **+** at the top right, then **New repository**.
2. Type the name `ci-practice`. Choose **Public**: the check reads it without signing in.
3. Add no README, `.gitignore` or licence.
4. Click **Create repository**.

In the terminal, use your account name. `git remote add` names your GitHub repository `origin`:

```sh
git remote add origin https://github.com/<your-account>/ci-practice.git
git push -u origin main
```

Expect a line that ends with `main -> main`.

## Step 4 · Ask Codex for the workflow, on a branch

```sh
git switch -c add-checks
codex --no-daemon
```

If your Codex does not know `--no-daemon`, leave it out. If Codex asks "Trust this folder?", choose `1. Trust and continue`. Type this request:

```text
Write .github/workflows/checks.yml. It runs on every pull request to main and every push to main. One job with the id checks and no name: line, on ubuntu-latest. Steps: actions/checkout, actions/setup-python with Python 3.12, pip install -r requirements.txt, then pytest. Change no other file.
```

Expect Codex to write one new file.

## Step 5 · Read the file before you accept it

Type `/diff` and compare the new file with this one. Press `q` to close it.

```yaml
name: checks                 # the workflow's name: any is fine
on:
  pull_request:              # each pull request
    branches:
      - main
  push:                      # each push, a merge too
    branches:
      - main
jobs:
  checks:                    # the job's id: the check's name
    runs-on: ubuntu-latest   # a fresh machine
    steps:
      - uses: actions/checkout@v6        # your committed files (@v6: the version)
      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'
      - run: pip install -r requirements.txt
      - run: pytest
```

The job `checks` has no `name:` line of its own. The `name:` at the top names the workflow, which is fine. Exercise 8 requires a check named exactly `checks`.

If a line is missing, tell Codex which one. Then type `/exit`.

## Step 6 · Push the branch, open a pull request, merge it when green

```sh
git add .github
git commit -m "Run the tests on every pull request"
git push -u origin add-checks
```

If the push refuses `checks.yml` and names the `workflow` scope, run `gh auth refresh --scopes workflow` and push again.

1. On your repository's page, click **Compare & pull request**, then **Create pull request**.
2. Wait about a minute for the check `checks`. If it fails, click **Actions**, the run, then the job: failed steps open by themselves. Ask Codex to fix what the log names, then commit and push. `checks` runs again.
3. When it passes, click **Merge pull request**, then `Confirm merge`.

Expect the pull request page to say that you merged it.

## Step 7 · Pull the merged main

```sh
git switch main
git pull
```

Expect `checks.yml` among the changed files. On github.com, click **Actions**. Expect a second run of `checks`, on `main`: the merge's push started it.

## Check your work

Write your repository in `REPO`, then run the cell. It reads GitHub twice at most, and saves the replies in `practice/github.json` for reruns. Expect two `ok:` lines, then `Done:`.

In [ ]:
# Check your work: checks.yml, then your newest pull request (2 public GitHub reads at most)
REPO = ""  # your repository, for example "your-account/ci-practice"
REFRESH = False  # True: ask GitHub again instead of reusing practice/github.json

import json
import re
import time
import urllib.error
import urllib.request
from pathlib import Path

WORKFLOW = Path("practice/.github/workflows/checks.yml")
SAVED = Path("practice/github.json")
AGAIN = " Then set REFRESH = True and rerun."
NEEDS = {
    "pull_request": r"pull_request", "push": r"\bpush\b", "main": r"\bmain\b", "job checks": r"(?m)^ +checks:\s*$",
    "ubuntu-latest": r"ubuntu-latest", "Python 3.12": r"python-version:\s*['\"]?3\.12\b",
    "requirements.txt": r"requirements\.txt", "pytest step": r"(?m)^\s*(- )?(run:\s*)?(python3? -m )?pytest\b",
}


def read(path: str, saved: dict) -> object:
    """Return GitHub's reply for path, saved or read now."""
    if path not in saved["reads"]:
        ask = urllib.request.Request("https://api.github.com" + path, headers={"User-Agent": "course-check"})
        try:
            with urllib.request.urlopen(ask, timeout=30) as reply:
                saved["reads"][path] = json.load(reply)
        except urllib.error.HTTPError as error:
            if error.code in (403, 429):
                reset = time.localtime(int(error.headers.get("x-ratelimit-reset") or time.time() + 3600))
                raise RuntimeError(f"GitHub allows 60 reads an hour per address, shared by your classroom. "
                                   f"Try again after {time.strftime('%H:%M', reset)}.")
            raise RuntimeError(f"GitHub answered {error.code} for {saved['repo']}. Check REPO, and that the "
                               "repository is public (Step 3).")
        except (urllib.error.URLError, OSError):
            raise RuntimeError("the check could not reach GitHub. Check your internet connection, then run it again.")
        SAVED.write_text(json.dumps(saved, indent=1), encoding="utf-8")
    return saved["reads"][path]


def first_problem() -> str:
    """Print ok for each check that passes; return the first problem, or ""."""
    if not WORKFLOW.is_file():
        return "practice/.github/workflows/checks.yml is missing. Ask Codex for it (Step 4), and pull main (Step 7)."
    text = WORKFLOW.read_text(encoding="utf-8")
    missing = [need for need, pattern in NEEDS.items() if not re.search(pattern, text)]
    if missing:
        return f"checks.yml has no {missing[0]}. Read it again (Step 5)."
    print("ok: checks.yml has the lines of Step 5")
    found = re.search(r"([\w.-]+)/([\w.-]+)$", REPO.strip())
    if not found:
        return "REPO is empty. Write your repository at the top of this cell."
    repo = f"{found[1]}/{found[2]}"
    saved = json.loads(SAVED.read_text(encoding="utf-8")) if SAVED.is_file() and not REFRESH else {}
    if saved.get("repo") != repo:
        saved = {"repo": repo, "reads": {}}
    pulls = read(f"/repos/{repo}/pulls?state=all&per_page=1", saved)
    if not pulls:
        return "your repository has no pull request. Open one (Step 6)." + AGAIN
    number, head = pulls[0]["number"], pulls[0]["head"]["sha"]
    runs = read(f"/repos/{repo}/commits/{head}/check-runs?check_name=checks", saved)["check_runs"]
    if not runs:
        return f"pull request #{number} has no check named checks. A name: line under the job renames it (Step 5)." + AGAIN
    if not any(run.get("conclusion") == "success" for run in runs):
        shown = runs[0].get("conclusion") or runs[0].get("status")
        return f"the checks check of pull request #{number} shows {shown}. Wait, or read its log (Step 6)." + AGAIN
    if not pulls[0].get("merged_at"):
        return f"pull request #{number} passed checks, but you did not merge it yet (Step 6)." + AGAIN
    print(f"ok: pull request #{number} passed checks, and you merged it")
    return ""


try:
    problem = first_problem()
except Exception as error:
    problem = str(error) if isinstance(error, RuntimeError) else f"the check stopped on {error!r}. Run Step 1 again."
print(f"Not yet: {problem}" if problem else
      "Done: GitHub tests each pull request to main, and one passed and merged.")

## What you learned

- A workflow tests each pull request and push to `main` on a fresh machine.
- That machine has only your commits and what the workflow installs.

**Reflect:** What does your own project need that is only on your laptop?